# 02 — LLM-as-a-Judge

**Needs `GROQ_API_KEY`.** (Free — see the folder README.)

---

## When string matching gives up

In notebook 01 every metric compared text to a fixed expected answer. That works
for `"144"` or `"Paris"`. But what's the 'expected string' for:

> *"Explain why the sky is blue."*

There are a million correct phrasings and no single gold string. `exact_match`
would score every one of them 0. Even token-F1 can't tell a *correct*
explanation from a fluent but *wrong* one.

The modern fix: **ask a strong LLM to grade the answer.** A capable model is
surprisingly good at judging correctness, relevance, and quality — far better
than any string metric on open-ended text. This is called **LLM-as-a-judge**,
and it's how teams evaluate chatbots, RAG systems, and agents today.

### What you'll build
1. A **binary judge** (correct / incorrect)
2. A **rubric judge** that scores 1–5 *and explains why*
3. A **multi-criteria judge** (correctness + relevance + coherence at once)
4. A **reference-free judge** (no gold answer available)
5. A **pairwise judge** (which of A vs B is better?)

...plus the **bias traps** that make naïve judges lie to you.

## Step 1 — Setup

Load the API key from the `.env` in the folder root and create the LLM client.
We use `temperature=0` everywhere: a judge should be **consistent**, not creative.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file (see the README)"
print("API key loaded.")

In [ ]:
from langchain_groq import ChatGroq

# temperature=0 -> the judge gives the same verdict every run (as much as possible)
judge = ChatGroq(model="llama3-8b-8192", temperature=0)
print(f"Judge LLM ready: {judge.model_name}")

## Step 2 — A safe JSON parser (you'll thank yourself later)

We'll ask the judge to reply in JSON so we can read its verdict programmatically.
But LLMs sometimes wrap JSON in prose ('Sure! Here is the result: {...}'). A small
helper that extracts the first `{...}` block and falls back gracefully saves you
from constant crashes.

In [ ]:
import json, re

def safe_parse(text: str) -> dict:
    """Pull the first JSON object out of an LLM response. Never raises."""
    # Grab the outermost {...} even if the model added chatter around it
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {"_raw": text, "_parse_error": True}

print(safe_parse('Here you go: {"score": 5, "reason": "great"} hope that helps!'))

## Step 3 — The binary judge (correct / incorrect)

The simplest judge: given a question, a reference answer, and the agent's answer,
decide **correct or not**. The key skills here are general to *all* judges:

- Give the judge a clear **role** and a **narrow question**.
- Show it the **reference** so it grades against truth, not its own opinion.
- Ask for **reasoning first, verdict second** — models judge better when they
  'think out loud' before committing (this is chain-of-thought, and it measurably
  improves judge accuracy).

In [ ]:
BINARY_PROMPT = """You are a strict grading assistant.
Decide whether the STUDENT ANSWER is correct, using the REFERENCE ANSWER as ground truth.
Minor wording differences are fine — judge the meaning, not the phrasing.

QUESTION: {question}
REFERENCE ANSWER: {reference}
STUDENT ANSWER: {answer}

First reason in one sentence, then give your verdict.
Respond ONLY as JSON: {{"reasoning": "...", "correct": true or false}}"""

def binary_judge(question: str, reference: str, answer: str) -> dict:
    prompt = BINARY_PROMPT.format(question=question, reference=reference, answer=answer)
    response = judge.invoke(prompt)
    return safe_parse(response.content)

# Same question, three different answers
q = "What is the capital of Australia?"
ref = "Canberra"
for ans in ["Canberra", "It's Canberra, the purpose-built capital.", "Sydney"]:
    verdict = binary_judge(q, ref, ans)
    print(f"answer={ans!r:55} -> correct={verdict.get('correct')}  ({verdict.get('reasoning','')[:60]})")

### Notice what just happened

All three answers would have scored differently under notebook 01's string
metrics. The judge correctly marks the first two **correct** (same meaning,
different words) and 'Sydney' **incorrect** — something `contains_match` and
`f1_score` could never reliably do. That's the power, and the reason judges have
taken over open-ended evaluation.

## Step 4 — The rubric judge (a 1–5 score with reasoning)

Binary is coarse. A **rubric** gives a graded score against *written criteria* —
the same way a teacher grades an essay. Spelling out what each score means makes
the judge far more consistent than just asking 'rate this 1–5'.

In [ ]:
RUBRIC_PROMPT = """You are an expert evaluator. Score the ANSWER from 1 to 5 using this rubric:
5 = fully correct and complete
4 = correct but missing a minor detail
3 = partially correct, or correct with a notable omission
2 = mostly incorrect, with a grain of truth
1 = completely incorrect or irrelevant

QUESTION: {question}
REFERENCE ANSWER: {reference}
ANSWER TO SCORE: {answer}

Reason briefly, then score.
Respond ONLY as JSON: {{"reasoning": "...", "score": <1-5>}}"""

def rubric_judge(question: str, reference: str, answer: str) -> dict:
    prompt = RUBRIC_PROMPT.format(question=question, reference=reference, answer=answer)
    return safe_parse(judge.invoke(prompt).content)

q = "What are the three primary colors of light?"
ref = "Red, green, and blue"
for ans in ["Red, green, and blue", "Red and green", "Red, yellow, and blue", "I'm not sure"]:
    v = rubric_judge(q, ref, ans)
    print(f"score={v.get('score')}  answer={ans!r:28}  -> {v.get('reasoning','')[:55]}")

## Step 5 — The multi-criteria judge

Real answers are good or bad along *several axes at once*. A response can be
factually correct but rambling and off-topic. We ask the judge to score multiple
**criteria** in a single call — efficient and gives you a richer scorecard:

- **correctness** — is it factually right vs the reference?
- **relevance** — does it actually answer the question asked?
- **coherence** — is it clear and well-structured?

In [ ]:
MULTI_PROMPT = """You are an evaluation expert. Score the ANSWER on three criteria, each 1-5:
- correctness: factually right vs the reference
- relevance:   directly addresses the question
- coherence:   clear, well-organized, easy to read

QUESTION: {question}
REFERENCE ANSWER: {reference}
ANSWER: {answer}

Respond ONLY as JSON:
{{"correctness": <1-5>, "relevance": <1-5>, "coherence": <1-5>, "reasoning": "..."}}"""

def multi_judge(question: str, reference: str, answer: str) -> dict:
    prompt = MULTI_PROMPT.format(question=question, reference=reference, answer=answer)
    return safe_parse(judge.invoke(prompt).content)

q = "Why do leaves turn green?"
ref = "Leaves are green because of chlorophyll, the pigment that absorbs sunlight for photosynthesis."
answer = "Leaves contain chlorophyll, a green pigment used to capture light energy for photosynthesis. Also, I love autumn and pumpkins!"

v = multi_judge(q, ref, answer)
for k in ["correctness", "relevance", "coherence"]:
    print(f"  {k:<12}: {v.get(k)}")
print(f"  reasoning : {v.get('reasoning','')}")

## Step 6 — The reference-free judge

Sometimes you have **no gold answer** — you're evaluating a brand-new question, or
a creative task with no single right response. The judge then grades on *intrinsic*
qualities: is the answer helpful, on-topic, and free of obvious errors? This is
how you score production traffic where nobody pre-wrote the answers.

> ⚠️ Reference-free judging is weaker — the model can't catch a confident factual
> error it also believes. Use it when you must, prefer a reference when you can.

In [ ]:
REF_FREE_PROMPT = """You are evaluating an AI assistant's answer. There is no reference answer.
Judge how helpful and appropriate the answer is for the question, 1-5:
5 = excellent, directly helpful and accurate-sounding
3 = partially helpful or generic
1 = unhelpful, evasive, or clearly wrong

QUESTION: {question}
ANSWER: {answer}

Respond ONLY as JSON: {{"reasoning": "...", "helpfulness": <1-5>}}"""

def reference_free_judge(question: str, answer: str) -> dict:
    prompt = REF_FREE_PROMPT.format(question=question, answer=answer)
    return safe_parse(judge.invoke(prompt).content)

q = "How can I make my morning routine more productive?"
for ans in ["Try waking up at a consistent time, plan your top 3 tasks the night before, and avoid your phone for the first 30 minutes.",
            "idk just google it"]:
    v = reference_free_judge(q, ans)
    print(f"helpfulness={v.get('helpfulness')}  ans={ans[:50]!r}")

## Step 7 — The pairwise judge (A vs B)

Often you don't need an absolute score — you need to know **which of two systems
is better**. Did my new prompt beat the old one? Pairwise comparison is what
powers LLM leaderboards (the 'Chatbot Arena' approach) and is the cleanest signal
when you're choosing between two versions (we use it for A/B testing in nb 05).

In [ ]:
PAIRWISE_PROMPT = """You are comparing two answers to the same question.
Decide which answer is better overall (more correct, relevant, and clear).

QUESTION: {question}

ANSWER A: {answer_a}

ANSWER B: {answer_b}

Reason briefly, then choose. A tie is allowed.
Respond ONLY as JSON: {{"reasoning": "...", "winner": "A" or "B" or "tie"}}"""

def pairwise_judge(question: str, answer_a: str, answer_b: str) -> dict:
    prompt = PAIRWISE_PROMPT.format(question=question, answer_a=answer_a, answer_b=answer_b)
    return safe_parse(judge.invoke(prompt).content)

q = "What is photosynthesis?"
a = "It's how plants make food."
b = "Photosynthesis is the process where plants convert sunlight, water, and CO2 into glucose and oxygen using chlorophyll."
v = pairwise_judge(q, a, b)
print(f"winner: {v.get('winner')}")
print(f"reason: {v.get('reasoning')}")

## Step 8 — Judges are biased. Here's how they lie.

LLM judges are powerful but **not neutral**. Two well-documented traps:

- **Position bias** — judges tend to favor whichever answer is shown *first* (or
  sometimes second). The fix: run the comparison **both ways** (A,B then B,A) and
  only trust a winner that holds in both orders.
- **Verbosity bias** — judges often rate *longer* answers higher even when the
  extra words add nothing. Watch for it; sometimes the terse answer is better.

Let's demonstrate position bias by swapping the order of the same two answers.

In [ ]:
def robust_pairwise(question: str, answer_a: str, answer_b: str) -> str:
    """Run the comparison in both orders to cancel out position bias."""
    first  = pairwise_judge(question, answer_a, answer_b).get("winner")   # A=answer_a, B=answer_b
    second = pairwise_judge(question, answer_b, answer_a).get("winner")   # A=answer_b, B=answer_a

    # Translate each verdict back to 'a' / 'b' / 'tie'
    pick1 = {"A": "a", "B": "b"}.get(first, "tie")
    pick2 = {"A": "b", "B": "a"}.get(second, "tie")  # order was swapped

    if pick1 == pick2:
        return pick1            # consistent winner — trustworthy
    return "tie (inconsistent)"  # flipped with order -> position bias, call it a tie

q = "Name a benefit of regular exercise."
a = "It improves cardiovascular health."
b = "It improves cardiovascular health and overall well-being."
print(f"order 1 winner: {pairwise_judge(q, a, b).get('winner')}")
print(f"order 2 winner: {pairwise_judge(q, b, a).get('winner')}")
print(f"robust verdict: {robust_pairwise(q, a, b)}")

## Step 9 — Judge a whole dataset

Just like notebook 01, the real value comes from running the judge over *many*
cases and aggregating. Here we score a small dataset with the rubric judge and
compute the average — a single number for 'how good is this agent on open-ended
questions?'

In [ ]:
qa_dataset = [
    {"q": "What causes the seasons?",          "ref": "The tilt of Earth's axis as it orbits the Sun.",            "ans": "Earth's axial tilt as it goes around the Sun."},
    {"q": "What is the boiling point of water?", "ref": "100 degrees Celsius at sea level.",                        "ans": "100C at sea level (lower at altitude)."},
    {"q": "Who painted the Mona Lisa?",         "ref": "Leonardo da Vinci.",                                       "ans": "Michelangelo."},
]

scores = []
for case in qa_dataset:
    v = rubric_judge(case["q"], case["ref"], case["ans"])
    score = v.get("score", 0)
    scores.append(score)
    print(f"score={score}  {case['q'][:35]:35} -> {case['ans'][:35]!r}")

avg = sum(scores) / len(scores)
print(f"\nAverage rubric score: {avg:.2f} / 5")

## Step 10 — Try it yourself

1. **Stress-test the judge.** Feed `binary_judge` a subtly wrong answer (right
   topic, wrong number) and see if it catches it.
2. **Tighten the rubric.** Edit `RUBRIC_PROMPT` so a 5 requires citing *why*, then
   re-run Step 9 and watch scores drop.
3. **Build your own criterion.** Add a `safety` score (1–5) to the multi-criteria
   judge and test it on a risky question.

In [ ]:
# Your playground — edit and re-run.
question = "What is the speed of light in a vacuum?"
reference = "Approximately 299,792 kilometers per second."
your_answer = "About 300,000 km/s."

print("binary:", binary_judge(question, reference, your_answer))
print("rubric:", rubric_judge(question, reference, your_answer))

## Summary

| What you built | When to reach for it |
|----------------|----------------------|
| `binary_judge` | Quick correct/incorrect on open-ended answers |
| `rubric_judge` | Graded 1–5 score against written criteria |
| `multi_judge` | Score correctness + relevance + coherence at once |
| `reference_free_judge` | No gold answer available (e.g. live traffic) |
| `pairwise_judge` | Choosing between two versions (A/B) |
| `robust_pairwise` | Pairwise *without* position bias |
| `safe_parse` | Never crash on messy LLM JSON |

**The big takeaway:** when no string can tell good from bad, a strong LLM can —
if you give it a clear role, show it the reference, ask for reasoning before the
verdict, and stay alert to position & verbosity bias.

**Next up → `03-tracing-and-observability`:** a score tells you *that* an answer
is bad. To find out *why*, you need to see every step the agent took. Time to
build a tracer.